# Musanga — NB04 V4.013 | Four-band spectral contribution to UAV Musanga detection

This experiment **does not train a new U-Net**. It asks whether all four MS bands (red, green, red-edge, NIR) provide additional species discrimination beyond local RGB statistics on the **same annotation-unit spatial hold-outs**. The reference for NB02 and NB03 is segmentation, so ROC-AUC values here are **not numerically comparable** with their pixel-F1, pixel-IoU, or cover R².

## Table of contents

- [00 — Setup and versioned inputs](#00--setup-and-versioned-inputs)
- [01 — Raster header quality audit](#01--raster-header-quality-audit)
- [02 — Spatial coverage audit](#02--spatial-coverage-audit)
- [03 — Coverage figure](#03--coverage-figure)
- [04 — Native-window QA](#04--native-window-qa)
- [05 — Alpha and duplicate-band QA](#05--alpha-and-duplicate-band-qa)
- [06 — Original annotation units](#06--original-annotation-units)
  - [06A — Geometry and frozen folds](#06a--geometry-and-frozen-folds)
  - [06B — Balanced crown/canopy sampling](#06b--balanced-crowncanopy-sampling)
- [07 — Four-band multispectral discrimination](#07--four-band-multispectral-discrimination)
  - [07A — Matched RGB/MS native-window extraction](#07a--matched-rgbms-native-window-extraction)
  - [07B — Within-acquisition spatial-fold tests](#07b--within-acquisition-spatial-fold-tests)
  - [07C — Cross-MOS spatial-fold transfer](#07c--cross-mos-spatial-fold-transfer)
- [08 — Frozen NB02 U-Net paired comparison](#08--frozen-nb02-u-net-paired-comparison)
  - [08A — Held-out NB02 probabilities](#08a--held-out-nb02-probabilities)
  - [08B — Frozen U-Net versus MS correction](#08b--frozen-u-net-versus-ms-correction)
- [09 — Seven-channel RGB + MS U-Net](#09--seven-channel-rgb--ms-u-net)
  - [09A — Strict paired RGB/MS chips](#09a--strict-paired-rgbms-chips)
  - [09B — NB02-initialized seven-channel training](#09b--nb02-initialized-seven-channel-training)
  - [09C — Paired held-out segmentation check](#09c--paired-held-out-segmentation-check)
- [10 — Interpretation and next-stage fractional cover](#10--interpretation-and-next-stage-fractional-cover)

**Quick run for NEW U-Net experiment:** 00 → 06A → 09A → 09B → 09C. Earlier sections 06B–08B are not required. GPU needed for training. New step 08 uses held-out NB02 checkpoints and requires a Colab GPU, does not retrain the U-Net, and caches each fold's sampled probabilities. Earlier QA sections 01–05 may be skipped if already run; running the notebook top-to-bottom is also valid.

**Strict experiment choices:** all four bands are mandatory for every paired observation. No 3-band variants. Compare RGB, MS raw digital numbers, MS relative spectra, RGB+MS raw, RGB+MS relative, with the **same frozen independent annotation units**. MOS 5–6 is included. No U-Net or DSM training here.

**Versioned outputs:** `Final/Outputs/v4/04_ms_qa_v4_013/`. All earlier NB04 outputs are preserved; V4.009 high-density samples can be reused after input-point and raster-file provenance checks. **Resumability:** 06A reconstructs in-memory geometries; 06B imports validated prior sampling points. **07A reuses fingerprint-verified V4.010/V4.011 completed measurements and saves each acquisition immediately** to a stable shared cache, so a restart or subsequent notebook version never requires repeating completed acquisitions. Rows are verified against current annotation points and raster metadata before reuse. Approximate **20 target positive and 20 target negative locations per annotation unit**, subject to crown area and actual valid raster footprint. This may take longer than V4.008, but every raster read is a small native window.

**Key validity distinction:** NB02 checkpoint for each row is the model whose outer held-out fold matches that row’s original plot/circle fold. The frozen NB02 probabilities are inferred only on their *own* held-out fold. A separate four-band MS classifier is trained using labels from the other folds, never from the test fold. A predeclared fixed 25% probability blend produces the exploratory combined prediction. No stacked meta-model is trained on outputs of other NB02 checkpoints, avoiding cross-checkpoint leakage. All variants use the same matched, valid four-band points.

**07A performance:** acquisition-level resume; spatially ordered 8×8 source reads; a larger bounded GDAL block cache. A checkpoint hit skips all expensive imagery reads. Every group reports **CACHE HIT**, **RECOVERED**, or **EXTRACT**.

**V4.013 adds a real early-fusion U-Net experiment:** NB02 pretrained RGB weights initialize a seven-channel U-Net, with zero-initialized spectral input weights; identical spatial folds, existing source-labelled training caches, strict inner split reuse and matched MS-valid masks. **This version's held-out segmentation check is chip based; full plot/circle fractional-cover evaluation follows only after training and alignment QA.**


## 00 — Setup and versioned inputs


In [ ]:
# 00 — Reuse NB00 inventory and frozen ROI; metadata only, no image read
from pathlib import Path
import json, math, re, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.warp import transform_bounds
from rasterio.vrt import WarpedVRT
from rasterio.transform import from_bounds
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
DATA=ROOT/'Data'
NB00=ROOT/'Outputs/v4/00_input_audit'
OUT=ROOT/'Outputs/v4/04_ms_qa_v4_013'
OUT.mkdir(parents=True,exist_ok=True)
REG=NB00/'00_ms_registry.json'
HEADERS=NB00/'00_ms_raster_headers.csv'
OVERLAP=NB00/'00_ms_rgb_extent_audit.csv'
assert REG.is_file() and HEADERS.is_file() and OVERLAP.is_file(),(
    'First run NB00 MS inventory cells; missing',REG,HEADERS,OVERLAP)
ms_manifest=json.loads(REG.read_text())
ms=pd.read_csv(HEADERS).fillna({'error':''})
overlap=pd.read_csv(OVERLAP)
ROI_PATH=DATA/'Ortho_RGB'/'ROI.shp'
roi=gpd.read_file(ROI_PATH)
assert roi.crs is not None and len(roi)>0
RGB_REFERENCE=ms_manifest['rgb_reference']
RGB_DIR=DATA/'Ortho_RGB'
RGB_FILES={group:RGB_DIR/name for group,name in RGB_REFERENCE.items()}
assert all(p.is_file() for p in RGB_FILES.values()),RGB_FILES
EDGE_DISTANCE_M=(1,3,5,10,20,30,50)
print('Available surveys:', sorted(ms.group.dropna().unique()))
print('NB00 MS records:',len(ms),'| paired extent records:',len(overlap))
display(ms[['group','band_hint','name','bands','dtype','nodata','error']].head(40))


## 01 — Raster header quality audit


In [ ]:
# 01 — Band headers only, ZERO source pixel reads
rows=[]
for rec in ms.itertuples():
    if getattr(rec,'error',''):continue
    path=Path(rec.path)
    if not path.is_file():continue
    with rasterio.Env(GDAL_CACHEMAX=32):
        with rasterio.open(path) as ds:
            for b in range(1,ds.count+1):
                rows.append(dict(group=rec.group,file=path.name,path=str(path),
                    band=b,band_hint=rec.band_hint,
                    description=ds.descriptions[b-1],
                    colorinterp=str(ds.colorinterp[b-1]),
                    dtype=ds.dtypes[b-1],scale=ds.scales[b-1],offset=ds.offsets[b-1],
                    nodata=ds.nodatavals[b-1],crs=str(ds.crs),
                    width=ds.width,height=ds.height,
                    pixel_x_m=abs(ds.transform.a),pixel_y_m=abs(ds.transform.e),
                    block_shape=str(ds.block_shapes[b-1]),
                    pixel_distribution='NOT SAMPLED',file_bytes=path.stat().st_size))
    print('HEADERS:',rec.group,rec.name,flush=True)
BAND_AUDIT=pd.DataFrame(rows)
assert not BAND_AUDIT.empty
BAND_AUDIT.to_csv(OUT/'04_ms_band_metadata_only.csv',index=False)
display(BAND_AUDIT[['group','file','band','band_hint','description','dtype',
                    'scale','offset','nodata','pixel_x_m','block_shape']])
print('No pixel data read, no band-value statistics inferred.')


## 02 — Spatial coverage audit


In [ ]:
# 02 — ROI / RGB / MS BOUNDING-BOX overlap ONLY, ZERO pixel reads
from shapely.geometry import box
from shapely.ops import unary_union
from rasterio.warp import transform_bounds
summary=[]
for group,records in ms.groupby('group',sort=True):
    if group not in RGB_FILES:continue
    with rasterio.open(RGB_FILES[group]) as rgb:
        study=unary_union(roi.to_crs(rgb.crs).geometry).intersection(box(*rgb.bounds))
        area=float(study.area)
        if area<=0:continue
        for rec in records.itertuples():
            if getattr(rec,'error',''):continue
            path=Path(rec.path)
            if not path.is_file():continue
            with rasterio.Env(GDAL_CACHEMAX=32):
                with rasterio.open(path) as src:
                    if src.crs is None:continue
                    bb=box(*transform_bounds(src.crs,rgb.crs,*src.bounds,densify_pts=21))
                    frac=float(study.intersection(bb).area/area)
                    summary.append(dict(group=group,file=rec.name,band_hint=rec.band_hint,
                        nominal_bands=int(src.count),roi_bbox_overlap_fraction=frac,
                        rgb_roi_area_ha=area/1e4,valid_pixel_coverage='NOT TESTED',
                        registration='NOT TESTED'))
            print('BBOX:',group,rec.name,round(frac,3),flush=True)
SUMMARY=pd.DataFrame(summary)
assert not SUMMARY.empty
SUMMARY.to_csv(OUT/'04_ms_rgb_roi_bbox_only.csv',index=False)
display(SUMMARY.sort_values(['group','file']).round(3))
print('Bounding-box overlap is only an upper limit on valid MS pixel coverage.')


## 03 — Coverage figure


In [ ]:
# 03 — Metadata coverage figure, NO pixel reads
d=SUMMARY.sort_values(['group','file'])
fig,ax=plt.subplots(figsize=(9,max(4,len(d)*.23)))
ax.barh(np.arange(len(d)),d.roi_bbox_overlap_fraction.to_numpy())
ax.set_yticks(np.arange(len(d)))
ax.set_yticklabels([f"{r.group}: {r.file[:40]}" for r in d.itertuples()],fontsize=7)
ax.invert_yaxis()
ax.set_xlim(0,1.02)
ax.set_xlabel('Bounding-box overlap with RGB ∩ ROI (not actual valid coverage)')
ax.set_title('MS inventory — metadata only')
fig.tight_layout()
fig.savefig(OUT/'04_ms_bbox_metadata_summary.png',dpi=120,bbox_inches='tight')
plt.show()
plt.close(fig)
print('DONE:',OUT)


## 04 — Native-window QA


In [ ]:
# 04 — STRICTLY BOUNDED native-window MS sampling; do not resample whole images.
# 25 files x at most 3 windows x <= 2/4 bands x 64x64 pixels.
# Full-width strip TIFFs may decode an entire scanline internally; never request
# hundreds of rows, use WarpedVRT, or keep open handles/arrays between files.
from rasterio.warp import transform as transform_coords
from rasterio.windows import Window
from shapely.geometry import Point,box
from shapely.ops import unary_union
import gc

WINDOW_SIZE=64
MAX_PATCHES_PER_FILE=3
QA_ROWS=[]
QA_PATCHES=[]
QA_WARNINGS=[]
for group,records in ms.groupby('group',sort=True):
    if group not in RGB_FILES:continue
    with rasterio.open(RGB_FILES[group]) as rgb:
        crs=rgb.crs
        study=unary_union(roi.to_crs(crs).geometry).intersection(box(*rgb.bounds))
        if study.is_empty:continue
        for rec in records.itertuples():
            if getattr(rec,'error',''):continue
            path=Path(rec.path)
            if not path.is_file():continue
            print('NATIVE WINDOWS:',group,path.name,flush=True)
            # Only one source open; cap GDAL's decompressed block cache at 16 MB.
            with rasterio.Env(GDAL_CACHEMAX=16):
                with rasterio.open(path) as src:
                    if src.crs is None:continue
                    from rasterio.warp import transform_bounds
                    source_box=box(*transform_bounds(src.crs,crs,*src.bounds,densify_pts=21))
                    relevant=study.intersection(source_box)
                    if relevant.is_empty:continue
                    bb=relevant.bounds
                    choices=[relevant.representative_point()]
                    # Deterministic geographic candidates, no raster samples/read.
                    for u,v in [(0.25,0.25),(0.75,0.75),(0.25,0.75),
                                (0.75,0.25),(0.5,0.5),(0.35,0.65),(0.65,0.35)]:
                        pt=Point(bb[0]+u*(bb[2]-bb[0]),bb[1]+v*(bb[3]-bb[1]))
                        if relevant.covers(pt):choices.append(pt)
                    seen=set()
                    count=0
                    for pt in choices:
                        if count>=MAX_PATCHES_PER_FILE:break
                        if src.crs==crs:x,y=pt.x,pt.y
                        else:
                            mx,my=transform_coords(crs,src.crs,[pt.x],[pt.y])
                            x,y=mx[0],my[0]
                        row,col=src.index(x,y)
                        if row<0 or col<0 or row>=src.height or col>=src.width:continue
                        col=int(np.clip(col-WINDOW_SIZE//2,0,max(0,src.width-WINDOW_SIZE)))
                        row=int(np.clip(row-WINDOW_SIZE//2,0,max(0,src.height-WINDOW_SIZE)))
                        key=(row//WINDOW_SIZE,col//WINDOW_SIZE)
                        if key in seen:continue
                        seen.add(key)
                        win=Window(col,row,min(WINDOW_SIZE,src.width-col),
                                   min(WINDOW_SIZE,src.height-row))
                        # Never read all bands at once; no large intermediate arrays.
                        pixel_arrays=[]
                        for band in range(1,src.count+1):
                            a=src.read(band,window=win,masked=True)
                            v=np.asarray(a.data)
                            valid=(~np.ma.getmaskarray(a))&np.isfinite(v)
                            vals=v[valid]
                            unique=np.unique(v) if v.size<=4096 else np.array([])
                            QA_ROWS.append(dict(group=group,file=path.name,
                                band_hint=rec.band_hint,band=band,patch=count,
                                row=row,col=col,n_window=int(v.size),
                                n_valid=int(valid.sum()),
                                valid_fraction=float(valid.mean()),
                                min=float(vals.min()) if len(vals) else np.nan,
                                p05=float(np.percentile(vals,5)) if len(vals) else np.nan,
                                median=float(np.median(vals)) if len(vals) else np.nan,
                                p95=float(np.percentile(vals,95)) if len(vals) else np.nan,
                                max=float(vals.max()) if len(vals) else np.nan,
                                zero_frac_valid=float(np.mean(vals==0)) if len(vals) else np.nan,
                                fraction_65535=float(np.mean(vals==65535)) if len(vals) else np.nan,
                                n_unique=int(len(unique)),
                                unique_values=str(unique[:12].tolist()) if len(unique)<=12 else 'many'))
                            # Caution: 2-band TIFF may be intensity + alpha. Report, do
                            # not pre-assign alpha without a diagnostic.
                            del a,v,valid,vals
                        count+=1
                    if count==0:QA_WARNINGS.append((group,path.name,'No in-footprint native patch'))
            print('  COMPLETED:',count,'native 64x64 patches; GDAL cache capped',flush=True)
            gc.collect()

PATCH_QA=pd.DataFrame(QA_ROWS)
assert not PATCH_QA.empty,'No valid sample windows in any MS raster'
PATCH_QA.to_csv(OUT/'04_native_patch_band_statistics.csv',index=False)
summ=(PATCH_QA.groupby(['group','file','band_hint','band'],dropna=False)
      .agg(n_patches=('patch','nunique'),mean_valid=('valid_fraction','mean'),
           median_p05=('p05','median'),median=('median','median'),
           median_p95=('p95','median'),mean_zero=('zero_frac_valid','mean'),
           mean_65535=('fraction_65535','mean'),
           median_n_unique=('n_unique','median')).reset_index())
summ.to_csv(OUT/'04_native_patch_summary.csv',index=False)
display(summ.round(3))
if QA_WARNINGS:print('NO VALID WINDOWS:',QA_WARNINGS)
print('Sample distributions are NOT proof of calibrated reflectance or registration.')
print('Native-window audit complete:',OUT)


## 05 — Alpha and duplicate-band QA


In [ ]:
# 05 — Alpha-aware paired spectral QA (MOS 5–6 red and MOS 1–2 red-edge duplicates)
# Read only 16x16 source windows, at most 36 coordinates per survey.
# A zero in band 1 with alpha == 0 is TRANSPARENT, not evidence of corruption.
from shapely.geometry import Point
from rasterio.warp import transform as warp_coords
from rasterio.windows import Window
from itertools import combinations
import gc
SAMPLE_N=6
SIZE=16
qa_rows=[]
groups_to_check=('MOS_5_6_MS','MOS_1_2_MS')
for group in groups_to_check:
    recs=ms[(ms.group==group) & (ms.band_hint.isin(['red','green','nir','red_edge']))].copy()
    assert not recs.empty
    rgb_path=RGB_FILES[group]
    with rasterio.open(rgb_path) as rgb:
        base_crs=rgb.crs
        study=unary_union(roi.to_crs(base_crs).geometry).intersection(box(*rgb.bounds))
        # Limit to MS union intersection: prevents wasting all samples outside imagery.
        boxes=[]
        for rec in recs.itertuples():
            with rasterio.open(rec.path) as src:
                boxes.append(box(*transform_bounds(src.crs,base_crs,*src.bounds,densify_pts=21)))
        footprint=study.intersection(unary_union(boxes))
        assert not footprint.is_empty,group
        xmin,ymin,xmax,ymax=footprint.bounds
        test_points=[]
        for fy in np.linspace(.08,.92,SAMPLE_N):
            for fx in np.linspace(.08,.92,SAMPLE_N):
                pt=Point(xmin+fx*(xmax-xmin),ymin+fy*(ymax-ymin))
                if footprint.covers(pt):test_points.append(pt)
        if not test_points:
            test_points=[footprint.representative_point()]
    print('ALPHA QA:',group,'candidate shared locations',len(test_points),flush=True)
    # Open only a single raster at a time. Save tiny arrays for cross-band QA.
    sampled={}
    for rec in recs.itertuples():
        collected={}
        with rasterio.Env(GDAL_CACHEMAX=8):
            with rasterio.open(rec.path) as src:
                for j,pt in enumerate(test_points):
                    if src.crs==base_crs: xx,yy=pt.x,pt.y
                    else:
                        x2,y2=warp_coords(base_crs,src.crs,[pt.x],[pt.y])
                        xx,yy=x2[0],y2[0]
                    row,col=src.index(xx,yy)
                    if row<SIZE or col<SIZE or row>=src.height-SIZE or col>=src.width-SIZE:continue
                    win=Window(col-SIZE//2,row-SIZE//2,SIZE,SIZE)
                    # Read band 1 and (when present) alpha-like band 2 only.
                    v=src.read(1,window=win,masked=True)
                    z=np.asarray(v.data)
                    valid=(~np.ma.getmaskarray(v)) & np.isfinite(z)
                    alpha=None
                    if src.count>=2:
                        a=src.read(2,window=win,masked=False)
                        alpha=np.asarray(a)
                        # Explicitly treat alpha=0 as invisible. Non-zero may include
                        # partial transparency; record it separately.
                        valid &= alpha>0
                    values=z[valid]
                    qa_rows.append(dict(group=group,file=rec.name,band=rec.band_hint,
                        location=j,valid_fraction=float(valid.mean()),
                        valid_pixels=int(valid.sum()),
                        alpha_zero_fraction=float(np.mean(alpha==0)) if alpha is not None else np.nan,
                        alpha_full_fraction=float(np.mean(alpha==255)) if alpha is not None else np.nan,
                        signal_median=float(np.median(values)) if len(values) else np.nan,
                        signal_zero_fraction=float(np.mean(values==0)) if len(values) else np.nan))
                    collected[j]=(z.copy(),valid.copy())
        sampled[rec.name]=collected
        print('  ',rec.name,'windows',len(collected),'valid locations',
              sum(mask.sum()>0 for z,mask in collected.values()),flush=True)
        gc.collect()
    if group=='MOS_1_2_MS':
        rededges=[p for p in recs[recs.band_hint.eq('red_edge')].name]
        if len(rededges)==2:
            a,b=(sampled[x] for x in rededges)
            shared=sorted(set(a)&set(b))
            different=0;valid_pixels=0
            for k in shared:
                za,ma=a[k];zb,mb=b[k]
                joint=ma&mb
                different+=int(np.count_nonzero(za[joint]!=zb[joint]))
                valid_pixels+=int(joint.sum())
            print('DUPLICATE RED EDGE: differing shared valid sampled pixels:',
                  different,'/',valid_pixels,'across',len(shared),'windows')
            if valid_pixels==0:
                print('  INCONCLUSIVE: no shared valid pixels sampled.')
            elif different==0:
                print('  Sampled values identical; full-file identity NOT proven.')
    del sampled
qa=pd.DataFrame(qa_rows)
qa.to_csv(OUT/'04_alpha_aware_paired_QA.csv',index=False)
aggregate=(qa.groupby(['group','band','file'],as_index=False).agg(
    n_windows=('location','nunique'),
    windows_with_valid=('valid_pixels',lambda v:int((v>0).sum())),
    mean_valid_fraction=('valid_fraction','mean'),
    mean_alpha_zero_fraction=('alpha_zero_fraction','mean'),
    median_signal=('signal_median','median')))
display(aggregate.round(3))
print('QA saved:',OUT/'04_alpha_aware_paired_QA.csv')
print('NOTE: these are sparse native windows, not an estimate of full-area valid coverage.')


**Quality-control interpretation (section 05)**

The previous cell 04 can report mean_valid=1 even when both spectral values and alpha are zero. Cell 05 explicitly checks the second channel as **candidate alpha**: pixels with zero in channel 2 are rejected. A high fraction of zero-alpha windows for MOS 5–6 red would indicate missing coverage, **not necessarily radiometric corruption**.

When valid red pixels exist, compare their paired local signals with green, NIR and red-edge, and then inspect RGB/MS alignment before computing spectral indices. MOS 1–2 duplicate red-edge filenames are tested at identical native locations; matching samples are not proof of full-file identity. No original dataset is modified.


## 06 — Original annotation units
,
,Only polygonal sources with frozen field-plot/circle fold assignments are included; the original GIS files are read-only. Area-changing repairs greater than 5% fail closed. All source annotations are counted, and no annotated unit may enter both train and validation.
,
,The geometry/unit inventory (06A) is reconstructed in memory; 06B and 07A are persistent, fingerprint-validated checkpoints. Run 00 → 06A → 06B → 07A after runtime restarts, but actual MS raster pixel reads only happen when inputs change or no matching checkpoint exists.

**V4.012:** Section 07A has stable cross-version per-acquisition caches and can recover verified complete tables from V4.010/V4.011. A partial or interrupted extraction only requires redoing the unfinished acquisition. The raw image and annotations are never overwritten.


### 06A — Geometry and frozen folds


In [ ]:
# 06A — All frozen exhaustive annotation units and per-acquisition coverage
from shapely.geometry import box, Point
from shapely.ops import unary_union
from shapely.strtree import STRtree
from rasterio.warp import transform_bounds
from rasterio.windows import Window
from rasterio.transform import rowcol
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import hashlib, gc, os
from collections import defaultdict
REF=ROOT/'Outputs/v4/01_reference_field78_combined'
CONTRACT=json.loads((REF/'01_field78_reference_contract.json').read_text())
NB00=json.loads(Path(CONTRACT['input_manifest']).read_text())
PLOTS=gpd.read_file(NB00['field_support_qa']).to_crs('EPSG:32635')
PF=pd.read_csv(CONTRACT['folds_csv'])
PLOTS=PLOTS.merge(PF[['source_id','site_key','cv_fold']],left_on='Plot_name',
                  right_on='source_id',validate='one_to_one')
CIRCLES=gpd.read_file(NB00['circle_complete_roi']).to_crs('EPSG:32635')
CF=pd.read_csv(REF/'01_field78_circle_folds.csv')
CIRCLES['circle_id']=np.arange(len(CIRCLES))
CIRCLES=CIRCLES.merge(CF[['circle_id','site_key','cv_fold',
                           'cross_fold_plot_intersection']],on='circle_id',validate='one_to_one')
CROWNS=gpd.read_file(NB00['exhaustive_plot_crowns']).to_crs('EPSG:32635')
CROWN_CIRC=gpd.read_file(NB00['circle_positive_crowns']).to_crs('EPSG:32635')
assert len(PLOTS)==78 and len(CIRCLES)==34
assert {'Class_ID'}.issubset(CROWNS.columns)
# Topology-safe polygon repair. Some digitized crowns have self-intersections.
# Only polygonal components are kept; never buffer(0) blindly or silently
# substitute an arbitrary boundary. Log repaired feature counts and area change.
from shapely import make_valid
from shapely.errors import GEOSException
def polygon_only(g):
    if g is None or g.is_empty:return None
    if g.geom_type in ('Polygon','MultiPolygon'):return g
    polygons=[]
    for z in getattr(g,'geoms',[]):
        p=polygon_only(z)
        if p is not None:polygons.append(p)
    return unary_union(polygons) if polygons else None
REPAIR_REPORT=[]
def safe_geom(g,label):
    if g is None or g.is_empty:return None
    old_area=float(g.area)
    if not g.is_valid:
        repaired=polygon_only(make_valid(g))
        if repaired is None or repaired.is_empty:
            raise RuntimeError(f'Cannot repair polygon {label}')
        change=abs(repaired.area-old_area)/max(old_area,1e-9)
        REPAIR_REPORT.append(dict(feature=label,area_before=old_area,
                                  area_after=float(repaired.area),
                                  area_rel_change=change))
        if change>.05:
            raise RuntimeError(f'Polygon repair changes area by {change:.1%}: {label}')
        g=repaired
    return g
for frame,name in [(PLOTS,'plot'),(CIRCLES,'circle'),
                   (CROWNS,'plot_crown'),(CROWN_CIRC,'circle_crown')]:
    frame.geometry=[safe_geom(g,f'{name}:{i}') for i,g in enumerate(frame.geometry)]
    if frame.geometry.isna().any():raise RuntimeError(f'Null geometry after repair: {name}')
pd.DataFrame(REPAIR_REPORT,columns=['feature','area_before','area_after',
   'area_rel_change']).to_csv(OUT/'06_polygon_repair_audit.csv',index=False)
print('Repaired',len(REPAIR_REPORT),'invalid polygon geometries; maximum area change',
      max([x['area_rel_change'] for x in REPAIR_REPORT],default=0))

# Use only geometry interior of existing exhaustive plot/circle; no partial crowns.
CROWNS=CROWNS.loc[CROWNS.geometry.notna()&~CROWNS.geometry.is_empty].copy()
CROWN_CIRC=CROWN_CIRC.loc[CROWN_CIRC.geometry.notna()&~CROWN_CIRC.geometry.is_empty].copy()
cgeoms=list(CROWNS.geometry)
ctree=STRtree(cgeoms)
ccgeoms=list(CROWN_CIRC.geometry)
cctree=STRtree(ccgeoms)
plot_geoms=list(PLOTS.geometry)
plot_tree=STRtree(plot_geoms)
RNG=np.random.default_rng(20261117)
SAMPLE_PER_CLASS_PER_UNIT=3
GUARD_M=15.0
NEW_OUT=OUT/'06_full_reference_unit_screen'
NEW_OUT.mkdir(parents=True,exist_ok=True)
GROUPS=['Baego_MS','MOS_1_2_MS','MOS_3_4_MS','MOS_5_6_MS',
        'Yangambi_YBJoseph_right_MS','Yangambi_YBForgeo_right_MS']
SITE_MAP={'Baego_MS':'Baego','MOS_1_2_MS':'MOS','MOS_3_4_MS':'MOS',
          'MOS_5_6_MS':'MOS','Yangambi_YBJoseph_right_MS':'Yangambi',
          'Yangambi_YBForgeo_right_MS':'Yangambi'}
MOSAIC_KEY={'Baego_MS':'Baego.tif',
            'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
            'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
            'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
            'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
            'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
def band_registry(group):
    part=ms[ms.group.eq(group)]
    if group=='Baego_MS':
        assert len(part)==1
        with rasterio.open(part.iloc[0].path) as ds:
            assert tuple(str(x).lower().replace(' ','') for x in ds.descriptions)==(
                'red','green','rededge','nir')
        return {b:Path(part.iloc[0].path) for b in ('red','green','red_edge','nir')}
    result={}
    for b in ('red','green','red_edge','nir'):
        cand=part[part.band_hint.eq(b)]
        if len(cand)>1 and b=='red_edge' and group=='MOS_1_2_MS':
            cand=cand[cand.name.eq('mlt_Mos_1_2_transparent_mosaic_red edge_georef.tif')]
        assert len(cand)==1,(group,b,list(cand.name))
        result[b]=Path(cand.iloc[0].path)
    return result
BANDS={group:band_registry(group) for group in GROUPS}
def bbox_for_group(group):
    boxes=[]
    rgbpath=DATA/'Ortho_RGB'/MOSAIC_KEY[group]
    with rasterio.open(rgbpath) as rgb:
        rgb_box=box(*rgb.bounds); rgbcrs=rgb.crs
    for p in set(BANDS[group].values()):
        with rasterio.open(p) as ds:
            boxes.append(box(*transform_bounds(ds.crs,rgbcrs,*ds.bounds,densify_pts=21)))
    footprint=rgb_box.intersection(unary_union(boxes)) # coarse bbox only
    return footprint,rgbcrs
UNIT_ROWS=[]
for group in GROUPS:
    footprint,crs=bbox_for_group(group)
    assert crs.to_epsg()==32635,(group,crs)
    site=SITE_MAP[group]
    for i,p in PLOTS[PLOTS.site_key.eq(site)].iterrows():
        geo=p.geometry
        if geo is None or geo.is_empty or not geo.intersects(footprint):continue
        region=geo.intersection(footprint)
        if region.is_empty:continue
        UNIT_ROWS.append(dict(group=group,site=site,kind='plot',uid=str(p.Plot_name),
            fold=int(p.cv_fold),plot_index=int(i),circle_index=-1,
            region=region,full_geometry=geo))
    for i,c in CIRCLES[CIRCLES.site_key.eq(site)].iterrows():
        geo=c.geometry
        if geo is None or geo.is_empty or not geo.intersects(footprint):continue
        if bool(c.cross_fold_plot_intersection):continue
        # Exclude any other intersection with field plot supervision.
        if len(plot_tree.query(geo,predicate='intersects')):continue
        region=geo.intersection(footprint)
        if region.is_empty:continue
        UNIT_ROWS.append(dict(group=group,site=site,kind='circle',
            uid=str(c.circle_id),fold=int(c.cv_fold),plot_index=-1,
            circle_index=int(i),region=region,full_geometry=geo))
units=pd.DataFrame(UNIT_ROWS)
assert not units.empty
units['unit_key']=units.apply(lambda r:f'{r.kind}:{r.uid}',axis=1)
assert not units.duplicated(['group','unit_key']).any()
print('ALL ELIGIBLE ORIGINAL UNITS IN MS BOUNDING FOOTPRINT:')
display(units.groupby(['group','kind','fold']).size().unstack(fill_value=0))
summary=units.drop(columns=['region','full_geometry'])
summary.to_csv(NEW_OUT/'04_all_annotation_units_bbox_audit.csv',index=False)
print('Note: these are bbox intersections; final valid-data count follows sampling.')


### 06B — Balanced crown/canopy sampling


In [ ]:
# 06B — Original annotation polygons: sample representative positive/negative positions
# Candidate points from every eligible plot or circle; avoid overlaying raw mosaics.
# Plot positives/negatives are the explicitly labelled Class_ID crown polygons.
# Circle negatives are remaining exhaustively labelled area excluding all positive crowns.
def point_candidates(region,n,seed):
    if region.is_empty:return []
    pts=[region.representative_point()]
    if len(pts)>=n:return pts
    xmin,ymin,xmax,ymax=region.bounds
    rng=np.random.default_rng(seed)
    # Draw multiple non-colocated interiors per annotated unit; cap attempts.
    for _ in range(max(300,n*100)):
        if len(pts)>=n:break
        p=Point(rng.uniform(xmin,xmax),rng.uniform(ymin,ymax))
        if region.contains(p) and all(p.distance(q)>=0.75 for q in pts):pts.append(p)
    return pts
def labelled_geometries(unit):
    area=unit['region']
    if unit['kind']=='plot':
        idx=ctree.query(unit['full_geometry'],predicate='intersects')
        pos=[];neg=[]
        for j in idx:
            g=cgeoms[int(j)].intersection(area)
            g=polygon_only(g)
            if g is None or g.is_empty or g.area==0:continue
            if int(CROWNS.iloc[int(j)].Class_ID)==1:pos.append(g)
            else:neg.append(g)
        return unary_union(pos) if pos else None,unary_union(neg) if neg else None
    idx=cctree.query(unit['full_geometry'],predicate='intersects')
    pos=[polygon_only(ccgeoms[int(j)].intersection(area)) for j in idx]
    pos=[g for g in pos if g is not None and not g.is_empty and g.area>0]
    p=unary_union(pos) if pos else None
    # Exclude a 0.5 m boundary buffer around positive crowns to reduce label ambiguity.
    neg=polygon_only(area.difference(p.buffer(.5))) if p is not None else area
    return p,neg
SAMPLE_PER_CLASS_PER_UNIT=20  # V4.009; deliberately larger than prior 3
# Candidate checkpoint: hash the exact repaired unit geometries, folds,
# plus sampling parameters so stale points cannot be silently reused.
from hashlib import sha256
SAMPLING_VERSION='v4_010_points_20_075m'
unit_digest=sha256(SAMPLING_VERSION.encode())
for rec in sorted(units.to_dict('records'),key=lambda r:(r['group'],r['unit_key'])):
    unit_digest.update(f"{rec['group']}|{rec['unit_key']}|{rec['fold']}|".encode())
    unit_digest.update(rec['region'].wkb)
POINTS_SIGNATURE=unit_digest.hexdigest()
POINTS_PATH=NEW_OUT/'04_all_unit_candidate_points.csv'
POINTS_SIG=NEW_OUT/'04_all_unit_candidate_points.sha256'
# Import the most recent fingerprint-matching candidate table, read-only.
# Same deterministic sampling procedure was used in V4.010 and V4.011.
if not (POINTS_PATH.is_file() and POINTS_SIG.is_file()
        and POINTS_SIG.read_text().strip()==POINTS_SIGNATURE):
    for prior_version in ('v4_011','v4_010'):
        prev=ROOT/'Outputs/v4'/('04_ms_qa_'+prior_version)/'06_full_reference_unit_screen'
        old=prev/'04_all_unit_candidate_points.csv'
        sig=prev/'04_all_unit_candidate_points.sha256'
        if old.is_file() and sig.is_file() and sig.read_text().strip()==POINTS_SIGNATURE:
            import shutil
            shutil.copyfile(old,POINTS_PATH)
            POINTS_SIG.write_text(POINTS_SIGNATURE)
            print('06B IMPORTED VERIFIED',prior_version,'POINT CACHE',flush=True)
            break
# Prefer own exact-signature cache; do not reuse mismatching old points.
POINTS_CACHE=False
if POINTS_PATH.is_file() and POINTS_SIG.is_file() and POINTS_SIG.read_text().strip()==POINTS_SIGNATURE:
    candidates=pd.read_csv(POINTS_PATH)
    POINTS_CACHE=True
    print('06B CHECKPOINT HIT:',POINTS_PATH,flush=True)
else:
    POINTS=[]
    for r in units.to_dict('records'):
        p,n=labelled_geometries(r)
        seed=int(hashlib.sha256(f"{r['group']}|{r['unit_key']}".encode()).hexdigest()[:8],16)
        for label,geo in [(1,p),(0,n)]:
            if geo is None or geo.is_empty or geo.area<.25:continue
            for pt in point_candidates(geo,SAMPLE_PER_CLASS_PER_UNIT,seed+label):
                POINTS.append(dict(group=r['group'],site=r['site'],kind=r['kind'],
                    unit_key=r['unit_key'],fold=r['fold'],
                    x=float(pt.x),y=float(pt.y),label=label))
    candidates=pd.DataFrame(POINTS)
    assert not candidates.empty
    candidates.to_csv(POINTS_PATH,index=False)
    POINTS_SIG.write_text(POINTS_SIGNATURE)
    print('06B CHECKPOINT SAVED:',POINTS_PATH,flush=True)
    print('Candidate points from original exhaustive annotations (max 20/class/unit):')
    display(candidates.groupby(['group','kind','fold','label']).agg(
        units=('unit_key','nunique'),points=('label','size')).reset_index())


## 07 — Four-band multispectral discrimination

All features are extracted at the same coordinates. Each fold holds out entire annotation units; a 15 m guard removes neighboring training units. *Absolute* denotes raw uncalibrated band digital numbers; *relative* denotes per-observation four-band spectral proportions and contrasts, **not calibrated reflectance indices**.


### 07A — Matched RGB/MS native-window extraction


In [ ]:
# 07A — Bounded native reads for RGB and each spectral band
# Read band 1 as spectral data for the individual-band sources.
# Ignore band 2 as a feature. Reject zero/nodata and honor raster masks.
# Every point uses an 8x8 native window; no whole-image read.
from rasterio.warp import transform as xy_transform
from rasterio.windows import Window
PATCH=8
def native_median(ds,x,y,band=1):
    rr,cc=ds.index(x,y)
    if rr<4 or cc<4 or rr>=ds.height-4 or cc>=ds.width-4:
        return np.nan,0.
    win=Window(cc-4,rr-4,PATCH,PATCH)
    z=ds.read(band,window=win,masked=True)
    a=np.asarray(z.data,np.float32)
    good=(~np.ma.getmaskarray(z))&np.isfinite(a)
    if ds.nodata is not None:good&=a!=ds.nodata
    good&=a!=0
    return (float(np.median(a[good])) if good.mean()>=.75 else np.nan,
            float(good.mean()))
# Persistent, per-acquisition resumable cache (V4.012).
# A complete V4.010/V4.011 extraction is recoverable without rereading TIFFs.
from hashlib import sha256
from pathlib import Path
from contextlib import ExitStack
import json, os, shutil, time
SAMPLE_PATH=NEW_OUT/'04_matched_original_units_ms.csv'
SAMPLE_SIG_PATH=NEW_OUT/'04_matched_original_units_ms_provenance.json'
SHARED=ROOT/'Outputs/v4/04_fourband_native_sample_cache'
SHARED.mkdir(parents=True,exist_ok=True)
RAW_BANDS=['MS_red','MS_green','MS_red_edge','MS_nir']
KEY=['group','kind','unit_key','fold','label','x','y']
FEATURES_REQUIRED=set(KEY+['site','R','G','B']+RAW_BANDS+['valid4'])
ALGORITHM='v4_010_native_median_8px_band1_nodata_exclude_zero'
def source_provenance_for_group(group):
    paths={Path(DATA/'Ortho_RGB'/MOSAIC_KEY[group])}
    paths.update(Path(p) for p in BANDS[group].values())
    out=[]
    for path in sorted(paths,key=str):
        st=path.stat()
        out.append([str(path),int(st.st_size),int(st.st_mtime_ns)])
    return out
def candidate_subset(group):
    return candidates.loc[candidates['group'].eq(group)].sort_values(
        KEY,kind='stable').reset_index(drop=True)
def group_signature(group):
    x=candidate_subset(group)
    h=sha256()
    h.update(ALGORITHM.encode())
    h.update(x[KEY].to_csv(index=False,float_format='%.12g').encode())
    h.update(json.dumps(source_provenance_for_group(group),sort_keys=True).encode())
    return h.hexdigest()
def exact_coverage(data,expected):
    if not FEATURES_REQUIRED.issubset(data.columns):return False
    if len(data)!=len(expected):return False
    a=data[KEY].copy().sort_values(KEY,kind='stable').reset_index(drop=True)
    b=expected[KEY].copy().sort_values(KEY,kind='stable').reset_index(drop=True)
    if a.duplicated().any() or b.duplicated().any():return False
    # Coordinates are saved in CSV; a 1e-6 m tolerance covers roundtripping.
    if not a[KEY[:-2]].astype(str).equals(b[KEY[:-2]].astype(str)):return False
    return np.allclose(a[['x','y']],b[['x','y']],rtol=0,atol=1e-6)
def validated_group_cache(group):
    tag=group.replace('/','_')
    csv=SHARED/(tag+'_native8.csv')
    meta=SHARED/(tag+'_native8.json')
    signature=group_signature(group)
    if csv.is_file() and meta.is_file():
        info=json.loads(meta.read_text())
        if info.get('signature')==signature:
            data=pd.read_csv(csv)
            if exact_coverage(data,candidate_subset(group)):
                print('07A CACHE HIT',group,'rows',len(data),flush=True)
                return data
            print('07A CACHE KEY/COVERAGE MISMATCH:',group,flush=True)
    return None
def save_group(group,data):
    assert exact_coverage(data,candidate_subset(group))
    tag=group.replace('/','_')
    path=SHARED/(tag+'_native8.csv')
    meta=SHARED/(tag+'_native8.json')
    temp=SHARED/(tag+'_native8.partial.csv')
    data.to_csv(temp,index=False)
    os.replace(temp,path)
    # Metadata written LAST, so incomplete jobs are never cache hits.
    mtmp=SHARED/(tag+'_native8.partial.json')
    mtmp.write_text(json.dumps({'group':group,'signature':group_signature(group),
                  'n':len(data),'algorithm':ALGORITHM,'source_provenance':
                  source_provenance_for_group(group)},indent=2))
    os.replace(mtmp,meta)
    print('07A CHECKPOINT SAVED',group,len(data),'rows',flush=True)
# Old complete notebook files can be imported if their original provenance
# (points + ALL original raster file stats + algorithm) matches EXACTLY.
def complete_signature_for_version(version):
    olddir=ROOT/'Outputs/v4'/('04_ms_qa_'+version)/'06_full_reference_unit_screen'
    oldpoints=olddir/'04_all_unit_candidate_points.csv'
    if not oldpoints.is_file():return None
    paths=set()
    for group in GROUPS:
        paths.add(Path(DATA/'Ortho_RGB'/MOSAIC_KEY[group]))
        paths.update(Path(p) for p in BANDS[group].values())
    prov=[]
    for p in sorted(paths,key=str):
        st=p.stat();prov.append([str(p),int(st.st_size),int(st.st_mtime_ns)])
    h=sha256(oldpoints.read_bytes())
    h.update(json.dumps(prov,sort_keys=True).encode())
    h.update(ALGORITHM.encode())
    return h.hexdigest()
prior_full={}
for ver in ('v4_011','v4_010'):
    root=ROOT/'Outputs/v4'/('04_ms_qa_'+ver)/'06_full_reference_unit_screen'
    f=root/'04_matched_original_units_ms.csv'
    m=root/'04_matched_original_units_ms_provenance.json'
    if not(f.is_file() and m.is_file()):continue
    meta=json.loads(m.read_text())
    if meta.get('signature')!=complete_signature_for_version(ver):
        print('07A ignoring old cache with changed input fingerprint:',ver,flush=True)
        continue
    olddata=pd.read_csv(f)
    if FEATURES_REQUIRED.issubset(olddata.columns):
        prior_full[ver]=olddata
        print('07A found validated complete prior extraction:',ver,len(olddata),'rows',flush=True)
OBS_GROUPS=[]
for group in GROUPS:
    expected=candidate_subset(group)
    if expected.empty:continue
    cached=validated_group_cache(group)
    if cached is not None:
        OBS_GROUPS.append(cached);continue
    # Historical complete tables may exclude points outside RGB boundary;
    # recover only groups with EXACT row-key coverage. Otherwise extract.
    recovered=False
    for ver,data in prior_full.items():
        sub=data.loc[data.group.eq(group)].copy()
        if exact_coverage(sub,expected):
            save_group(group,sub)
            OBS_GROUPS.append(sub)
            print('07A RECOVERED',group,'from',ver,flush=True)
            recovered=True;break
    if recovered:continue
    print('07A EXTRACT',group,'candidate points',len(expected),flush=True)
    tic=time.monotonic()
    # GDAL cache increased from 12MB to 256MB to reduce repeated TIFF strip
    # decompression for dispersed 8x8 window reads.
    with rasterio.Env(GDAL_CACHEMAX=256):
        with ExitStack() as stack:
            rgb=stack.enter_context(rasterio.open(DATA/'Ortho_RGB'/MOSAIC_KEY[group]))
            sources={str(path):stack.enter_context(rasterio.open(path))
                     for path in set(BANDS[group].values())}
            # Order points in native RGB row-major order to improve strip locality.
            indexed=[]
            for row in expected.itertuples(index=False):
                r,c=rgb.index(float(row.x),float(row.y))
                indexed.append((r,c,row))
            indexed.sort(key=lambda t:(t[0],t[1]))
            rows=[]
            for j,(rr,cc,row) in enumerate(indexed):
                if rr<4 or cc<4 or rr>=rgb.height-4 or cc>=rgb.width-4:
                    # Keep a transparent invalid record to preserve exact
                    # candidate coverage and avoid repeated slow cache misses.
                    record=dict(group=group,site=row.site,kind=row.kind,
                        unit_key=row.unit_key,fold=int(row.fold),label=int(row.label),
                        x=float(row.x),y=float(row.y),
                        R=np.nan,G=np.nan,B=np.nan)
                    for band in ('red','green','red_edge','nir'):
                        record['MS_'+band]=np.nan
                        record['valid_'+band]=0.
                    rows.append(record);continue
                arr=rgb.read([1,2,3],window=Window(cc-4,rr-4,PATCH,PATCH),masked=True)
                data=np.asarray(arr.data,np.float32)
                mask=(~np.ma.getmaskarray(arr).any(axis=0))
                rgbvals=np.median(data[:,mask],axis=1)/255. if mask.mean()>=.75 else [np.nan]*3
                record=dict(group=group,site=row.site,kind=row.kind,
                    unit_key=row.unit_key,fold=int(row.fold),label=int(row.label),
                    x=float(row.x),y=float(row.y),
                    R=rgbvals[0],G=rgbvals[1],B=rgbvals[2])
                for band in ('red','green','red_edge','nir'):
                    ds=sources[str(BANDS[group][band])]
                    xx,yy=float(row.x),float(row.y)
                    if ds.crs!=rgb.crs:
                        tx,ty=xy_transform(rgb.crs,ds.crs,[xx],[yy])
                        xx,yy=tx[0],ty[0]
                    idx={'red':1,'green':2,'red_edge':3,'nir':4}[band] if group=='Baego_MS' else 1
                    value,fraction=native_median(ds,xx,yy,idx)
                    record['MS_'+band]=value
                    record['valid_'+band]=fraction
                rows.append(record)
                if (j+1)%300==0:
                    print('07A EXTRACT',group,j+1,'/',len(indexed),flush=True)
    df=pd.DataFrame(rows)
    df['valid4']=df[['MS_red','MS_green','MS_red_edge','MS_nir']].notna().all(axis=1)
    save_group(group,df)
    OBS_GROUPS.append(df)
    print('07A DONE',group,'elapsed_seconds',round(time.monotonic()-tic,1),flush=True)
observed=pd.concat(OBS_GROUPS,ignore_index=True)
assert not observed.empty
# Every candidate belongs to exactly one recovered or extracted row.
assert sum(len(x) for x in OBS_GROUPS)==len(candidates)
assert observed[KEY].duplicated().sum()==0
# Durable combined export for diagnostics; per-group cache is the restart key.
observed.to_csv(SAMPLE_PATH,index=False)
SAMPLE_SIG_PATH.write_text(json.dumps({'schema':'v4_012_resume_per_acquisition',
   'n':len(observed),'group_signatures':{g:group_signature(g) for g in GROUPS}},indent=2))
print('07A ALL GROUPS COMPLETE',len(observed),'rows; four-band complete',
    int(observed[RAW_BANDS].notna().all(axis=1).sum()),flush=True)
display(observed.groupby(['group','kind','fold']).agg(
    units=('unit_key','nunique'),points=('label','size'),
    good4=('valid4','sum')).reset_index())


### 07B — Within-acquisition spatial-fold tests


In [ ]:
# 07B — Four-band MS discrimination; same matched rows for EVERY model
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from shapely.ops import unary_union
import numpy as np, pandas as pd

BASE=['R','G','B']
RAW=['MS_red','MS_green','MS_red_edge','MS_nir']
# Use only physically nonnegative four-band values.
Q=observed.loc[observed[RAW+BASE].notna().all(axis=1)].copy()
Q=Q.loc[(Q[RAW].to_numpy(float)>0).all(axis=1)].copy()
s=Q[RAW].to_numpy(np.float64)
den=s.sum(axis=1)
assert np.all(den>0)
for i,name in enumerate(('red','green','red_edge','nir')):
    Q['rel_'+name]=s[:,i]/den
# Four fractions sum to one; pairwise DN contrasts help express shape.
for a,b in [('nir','red'),('nir','red_edge'),('red_edge','red'),('green','red')]:
    Q['contrast_'+a+'_'+b]=(Q['MS_'+a]-Q['MS_'+b])/(Q['MS_'+a]+Q['MS_'+b])
REL=['rel_red','rel_green','rel_red_edge','rel_nir',
     'contrast_nir_red','contrast_nir_red_edge',
     'contrast_red_edge_red','contrast_green_red']
FEATURE_SETS={'RGB':BASE,'MS_absolute':RAW,'MS_relative':REL,
              'RGB_MS_absolute':BASE+RAW,'RGB_MS_relative':BASE+REL}
assert len(Q)>0
# Give every unit equal *total* weight, and where possible give Musanga
# and non-Musanga equal weight inside each unit.
def unit_weights(df):
    w=np.zeros(len(df),np.float64)
    for _,idx in df.groupby('unit_key').indices.items():
        sub=df.iloc[idx]
        for label,arr in sub.groupby('label').indices.items():
            # groupby indices relative to sub
            w[idx[np.asarray(arr,dtype=int)]] = 1./(sub.label.nunique()*len(arr))
    return w / max(np.mean(w),1e-12)
# Training and validation both unit-balanced: nearby pixels are not independent.
def fit_score(train,val,cols):
    model=HistGradientBoostingClassifier(max_iter=70,max_leaf_nodes=8,
             max_depth=3,learning_rate=.06,min_samples_leaf=5,
             l2_regularization=4,early_stopping=False,random_state=20261117)
    model.fit(train[cols].to_numpy(float),train['label'].to_numpy(int),
              sample_weight=unit_weights(train))
    pred=model.predict_proba(val[cols].to_numpy(float))[:,1]
    return float(roc_auc_score(val['label'].to_numpy(int),pred,
                sample_weight=unit_weights(val)))
def fold_partition(group,fold,data):
    val=data.loc[(data.group==group)&(data.fold==fold)].copy()
    train=data.loc[(data.group==group)&(data.fold!=fold)].copy()
    valunits=units.loc[(units.group==group)&(units.fold==fold)]
    other=units.loc[(units.group==group)&(units.fold!=fold)]
    if len(valunits) and len(other) and len(train):
        blocked=unary_union([geo.buffer(GUARD_M) for geo in valunits.full_geometry])
        bad=set(other.loc[other.full_geometry.map(lambda g:g.intersects(blocked)).to_numpy(bool),'unit_key'])
        train=train.loc[~train.unit_key.isin(bad)].copy()
    return train,val
def fold_sufficient(train,val):
    if train.empty or val.empty:return False,'empty training or test fold'
    if train.unit_key.nunique()<4 or val.unit_key.nunique()<2:return False,'few independent units'
    if train.label.nunique()<2 or val.label.nunique()<2:return False,'missing annotated class'
    if len(train)<20 or len(val)<8:return False,'too few sampled points'
    if not set(train.unit_key).isdisjoint(set(val.unit_key)):raise RuntimeError('unit leakage')
    return True,'evaluated'
RESULT_ROWS=[]
for group in GROUPS:
    for fold in range(3):
        tr,va=fold_partition(group,fold,Q)
        ok,why=fold_sufficient(tr,va)
        record=dict(group=group,fold=fold,status=why,
            train_units=tr.unit_key.nunique(),val_units=va.unit_key.nunique(),
            n_train=len(tr),n_val=len(va))
        if ok:
            for name,cols in FEATURE_SETS.items():
                record['auc_'+name]=fit_score(tr,va,cols)
            for variant in ('absolute','relative'):
                record['delta_'+variant]=record['auc_RGB_MS_'+variant]-record['auc_RGB']
        RESULT_ROWS.append(record)
        print('SPATIAL',group,'fold',fold,why,
             ('Δraw='+str(round(record['delta_absolute'],3))+
              ' Δrel='+str(round(record['delta_relative'],3))) if ok else '',flush=True)
RESULTS=pd.DataFrame(RESULT_ROWS)
RESULTS.to_csv(NEW_OUT/'04_v010_within_acquisition_fourband.csv',index=False)
display(RESULTS.round(3))
display(RESULTS.loc[RESULTS.status.eq('evaluated')].groupby('group').agg(
    folds=('fold','size'),delta_absolute=('delta_absolute','mean'),
    delta_relative=('delta_relative','mean')).round(3))
print('Spatial-unit-weighted ROC-AUC on each matched hold-out; NB02/NB03 segmentation metrics are DIFFERENT.')


### 07C — Cross-MOS spatial-fold transfer


In [ ]:
# 07C — Pooled MOS 1–2 / 3–4 / 5–6, frozen held-out folds
# Compare RGB vs FOUR-BAND ABSOLUTE and RELATIVE MS only on identical support.
# A single group identifier is NOT used as a feature. All independent
# plot/circle units from the same frozen fold are held out together.
MOS_GROUPS=['MOS_1_2_MS','MOS_3_4_MS','MOS_5_6_MS']
mos=Q.loc[Q.group.isin(MOS_GROUPS)].copy()
mos['spatial_id']=mos.group.astype(str)+'|'+mos.unit_key.astype(str)
M=[]
for fold in range(3):
    val=mos.loc[mos.fold.eq(fold)].copy()
    train=mos.loc[mos.fold.ne(fold)].copy()
    held=units.loc[units.group.isin(MOS_GROUPS)&units.fold.eq(fold)]
    other=units.loc[units.group.isin(MOS_GROUPS)&units.fold.ne(fold)]
    if len(held) and len(other) and len(train):
        buffer=unary_union([g.buffer(GUARD_M) for g in held.full_geometry])
        rejected=set((other.loc[other.full_geometry.map(
            lambda g:g.intersects(buffer)).to_numpy(bool),'group'].astype(str)+'|'+
            other.loc[other.full_geometry.map(
            lambda g:g.intersects(buffer)).to_numpy(bool),'unit_key'].astype(str)))
        train=train.loc[~train.spatial_id.isin(rejected)].copy()
    # unit_weights uses unit_key: MOS plot IDs may repeat across acquisition;
    # explicitly prefix to ensure equal weight PER acquisition-unit.
    train['unit_key']=train['spatial_id']
    val['unit_key']=val['spatial_id']
    ok,why=fold_sufficient(train,val)
    z=dict(fold=fold,status=why,train_units=train.unit_key.nunique(),
           val_units=val.unit_key.nunique(),n_train=len(train),n_val=len(val))
    if ok:
        for name,cols in FEATURE_SETS.items():z['auc_'+name]=fit_score(train,val,cols)
        z['delta_absolute']=z['auc_RGB_MS_absolute']-z['auc_RGB']
        z['delta_relative']=z['auc_RGB_MS_relative']-z['auc_RGB']
    M.append(z)
    print('CROSS-MOS fold',fold,why,flush=True)
MOS_POOLED=pd.DataFrame(M)
display(MOS_POOLED.round(3))
MOS_POOLED.to_csv(NEW_OUT/'04_v010_pooled_MOS_fourband.csv',index=False)
print('Pooled model is a transfer-screen only; systematic flight-specific radiometric shifts remain a confound.')


## 08 — Frozen NB02 U-Net paired comparison

This step tests whether MS carries information **beyond the actual frozen NB02 RGB U-Net**, rather than beyond another RGB histogram-boosting classifier. It computes each sample’s NB02 probability with the checkpoint corresponding to that sample’s **outer held-out fold**. The original NB02 U-Net is never modified.

A true paired evaluation compares (i) raw NB02 out-of-fold probability, (ii) a calibration model with NB02 probability alone and (iii) a correction model with NB02 probability + normalized four-band MS, on identical held-out annotation-unit observations. Corrections are trained only on **out-of-fold** NB02 predictions from the other folds, guarded from spatially neighboring units.

**Implementation:** original 256×256 / 128-pixel-stride Hann-blended NB02 grid, but evaluates only the tile pixels required for sampled points. Each unique tile is inferred once per fold and mosaic. Bounded GPU batches and read-only source rasters. Results are cached in the V4.011 output folder. This is not a full segmentation-map comparison; final F1/IoU and fractional-cover metrics still require support-matched dense predictions.


### 08A — Held-out NB02 probabilities


In [ ]:
# 08A — Extract frozen NB02 probabilities at NB04 four-band labelled positions
# Run after 07B; Q contains identical four-band-complete matched sample locations.
from pathlib import Path
from collections import defaultdict
import gc, json, hashlib, os
import torch
import importlib.util, subprocess, sys
if importlib.util.find_spec('segmentation_models_pytorch') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','segmentation-models-pytorch'])
import segmentation_models_pytorch as smp
from rasterio.windows import Window
from rasterio.transform import rowcol
from contextlib import ExitStack
torch.set_num_threads(2)
BASELINE_DIR=ROOT/'Outputs/v4/02_rgb_unet_field78_v4_007_full'
RGB_CKPT={f:BASELINE_DIR/f'NB02_V4_007_fold{f}_best.pt' for f in (0,1,2)}
for p in RGB_CKPT.values():assert p.is_file(),('Frozen NB02 checkpoint missing',p)
DEVICE_NB02=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE_NB02.type=='cuda','Select GPU runtime to avoid impractically slow repeated NB02 inference.'
TILE=256;STRIDE=128
assert TILE==2*STRIDE
hanning=np.hanning(TILE+2)[1:-1].astype(np.float32)
HANN=np.maximum(np.outer(hanning,hanning),.01).astype(np.float32)
HANN/=HANN.max()
BATCH=8
PROB_DIR=OUT/'08_nb02_ms_bridge'
PROB_DIR.mkdir(parents=True,exist_ok=True)
RAW=['MS_red','MS_green','MS_red_edge','MS_nir']
BASE=['R','G','B']
selected=Q.loc[Q[BASE+RAW].notna().all(axis=1)].copy()
selected=selected.loc[(selected[RAW]>0).all(axis=1)].copy()
assert not selected.empty
# Create stable row identities across runtime restarts and sample ordering.
selected['sample_id']=[hashlib.sha256(
    f"{r.group}|{r.unit_key}|{int(r.fold)}|{int(r.label)}|{r.x:.6f}|{r.y:.6f}".encode()).hexdigest()[:24]
    for r in selected.itertuples(index=False)]
assert selected.sample_id.is_unique, 'Duplicate sample identities; check candidate sampling.'
selected=selected.sort_values('sample_id').reset_index(drop=True)
# Exact provenance requires same sample list and original checkpoint bytes' metadata.
state=[]
for p in RGB_CKPT.values():
    z=p.stat();state.append((str(p),z.st_size,z.st_mtime_ns))
for g in GROUPS:
    p=DATA/'Ortho_RGB'/MOSAIC_KEY[g];z=p.stat()
    state.append((str(p),z.st_size,z.st_mtime_ns))
fp=hashlib.sha256()
fp.update(selected[['sample_id','group','fold','x','y']].to_csv(index=False).encode())
fp.update(json.dumps(state,sort_keys=True).encode())
fp.update(b'NB02-v4.007-global-Hann256-stride128-v011')
FINGERPRINT=fp.hexdigest()
cache=PROB_DIR/'08_nb02_heldout_probabilities.csv'
metadata=PROB_DIR/'08_nb02_heldout_probabilities.json'
if cache.is_file() and metadata.is_file() and json.loads(metadata.read_text()).get('fingerprint')==FINGERPRINT:
    old=pd.read_csv(cache)
    assert old.sample_id.is_unique and set(old.sample_id)==set(selected.sample_id)
    pred=selected.merge(old[['sample_id','nb02_prob']],on='sample_id',validate='one_to_one')
    assert pred.nb02_prob.between(0,1).all()
    print('NB02 PROBABILITY CACHE HIT:',len(pred),'sample locations',flush=True)
else:
    # Read only native RGB windows for required 256x256 global grid positions.
    # For a pixel, only 2x2 128px-stride tiles contribute positive Hann weight.
    allpred=[]
    def load_rgb_checkpoint(fold):
        state=torch.load(RGB_CKPT[fold],map_location='cpu',weights_only=False)
        assert state['version']=='V4.007' and int(state['fold'])==fold
        net=smp.Unet(encoder_name='resnet18',encoder_weights=None,
                     in_channels=3,classes=1).to(DEVICE_NB02)
        weights=state['model']
        assert all(k.startswith('net.') for k in weights)
        net.load_state_dict({k[4:]:v for k,v in weights.items()},strict=True)
        return net.eval().requires_grad_(False)
    for fold in (0,1,2):
        net=load_rgb_checkpoint(fold)
        subset=selected.loc[selected.fold.eq(fold)]
        for group,fr in subset.groupby('group',sort=True):
            if fr.empty:continue
            rgbpath=DATA/'Ortho_RGB'/MOSAIC_KEY[group]
            with rasterio.Env(GDAL_CACHEMAX=24):
                with rasterio.open(rgbpath) as ds:
                    coords=[]
                    grid=defaultdict(list)
                    for r in fr.itertuples(index=False):
                        rr,cc=ds.index(float(r.x),float(r.y))
                        assert 0<=rr<ds.height and 0<=cc<ds.width,(r.sample_id,rr,cc)
                        coords.append((r.sample_id,rr,cc))
                        # Adjacent anchored starts cover the target exactly.
                        rs=[((rr//STRIDE)-1)*STRIDE,(rr//STRIDE)*STRIDE]
                        cs=[((cc//STRIDE)-1)*STRIDE,(cc//STRIDE)*STRIDE]
                        for a in rs:
                            for b in cs:
                                if a<0 or b<0 or a>=ds.height or b>=ds.width:continue
                                grid[(a,b)].append((r.sample_id,rr-a,cc-b))
                    sums=defaultdict(float);weightsums=defaultdict(float)
                    keys=sorted(grid)
                    for st in range(0,len(keys),BATCH):
                        batchkeys=keys[st:st+BATCH]
                        xx=[]
                        for a,b in batchkeys:
                            # Match original NB02 zero-padded edge convention.
                            rgb=ds.read([1,2,3],window=Window(b,a,TILE,TILE),
                                        boundless=True,fill_value=0).astype(np.float32)/255.
                            xx.append(rgb)
                        batch=torch.from_numpy(np.stack(xx)).to(DEVICE_NB02)
                        with torch.inference_mode(),torch.autocast(
                            device_type='cuda',dtype=torch.float16):
                            prob=torch.sigmoid(net(batch)).float().cpu().numpy()[:,0]
                        for i,key in enumerate(batchkeys):
                            for sid,ir,ic in grid[key]:
                                if not (0<=ir<TILE and 0<=ic<TILE):continue
                                wt=float(HANN[ir,ic])
                                sums[sid]+=float(prob[i,ir,ic])*wt
                                weightsums[sid]+=wt
                        del batch,prob,xx
                        if (st//BATCH)%50==0:
                            print('NB02 OOF',fold,group,st+len(batchkeys),'/',len(keys),
                                  'unique tiles',flush=True)
                    for sid,rr,cc in coords:
                        assert weightsums[sid]>0,(fold,group,sid)
                        allpred.append(dict(sample_id=sid,nb02_prob=sums[sid]/weightsums[sid]))
            print('NB02 done:',fold,group,'samples',len(fr),'tiles',len(keys),flush=True)
        del net
        torch.cuda.empty_cache();gc.collect()
    old=pd.DataFrame(allpred)
    assert len(old)==len(selected) and old.sample_id.is_unique
    pred=selected.merge(old,on='sample_id',validate='one_to_one')
    assert pred.nb02_prob.between(0,1).all()
    tmp=cache.with_suffix('.tmp.csv')
    old.to_csv(tmp,index=False);os.replace(tmp,cache)
    metadata.write_text(json.dumps({'fingerprint':FINGERPRINT,
            'n_samples':len(pred),'source_model':'NB02_V4_007_frozen_outer_fold',
            'tile':TILE,'stride':STRIDE},indent=2))
    print('NB02 OOF PROBABILITIES SAVED:',cache,flush=True)
display(pred.groupby(['group','fold']).agg(units=('unit_key','nunique'),
        samples=('sample_id','size'),positive=('label','sum'),
        mean_nb02=('nb02_prob','mean')).reset_index().round(3))


### 08B — Frozen U-Net versus MS correction


In [ ]:
# 08B — Strict outer-fold test: no trainable stacking on NB02 probabilities.
# Compare frozen NB02, a MS-only four-band classifier trained on the OTHER
# original units, and a fixed 75% NB02 + 25% MS blend.
# Pre-specified alpha=0.25 (no fold-specific tuning on held-out outcomes).
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from shapely.ops import unary_union
import numpy as np,pandas as pd
ALPHA_MS=0.25
assert 0<ALPHA_MS<1
C=pred.copy()
raw=C[['MS_red','MS_green','MS_red_edge','MS_nir']].to_numpy(np.float64)
den=raw.sum(axis=1)
assert (den>0).all()
for i,b in enumerate(('red','green','red_edge','nir')):
    C['rel_'+b]=raw[:,i]/den
for a,b in [('nir','red'),('nir','red_edge'),('red_edge','red'),('green','red')]:
    C['contrast_'+a+'_'+b]=(C['MS_'+a]-C['MS_'+b])/(C['MS_'+a]+C['MS_'+b])
REL=['rel_red','rel_green','rel_red_edge','rel_nir',
     'contrast_nir_red','contrast_nir_red_edge','contrast_red_edge_red','contrast_green_red']
def unit_weights(frame):
    w=np.zeros(len(frame),dtype=float)
    for _,idx in frame.groupby('eval_id').indices.items():
        z=frame.iloc[idx]
        for _,ids in z.groupby('label').indices.items():
            w[np.asarray(idx)[np.asarray(ids,int)]]=1/(z.label.nunique()*len(ids))
    assert (w>0).all()
    return w/np.mean(w)
def metrics(frame,prob):
    y=frame.label.to_numpy(int);w=unit_weights(frame)
    return dict(auc=float(roc_auc_score(y,prob,sample_weight=w)),
      brier=float(np.average((prob-y)**2,weights=w)))
def train_ms_classifier(train,test):
    model=HistGradientBoostingClassifier(max_iter=70,max_depth=3,max_leaf_nodes=7,
       min_samples_leaf=8,learning_rate=.05,l2_regularization=5,
       early_stopping=False,random_state=20261117)
    model.fit(train[REL],train.label,sample_weight=unit_weights(train))
    return model.predict_proba(test[REL])[:,1]
RESULTS=[]
for scope in ['MOS_pooled','Yangambi_YBJoseph_right_MS','Baego_MS']:
    if scope=='MOS_pooled':
        q=C.loc[C.group.isin(['MOS_1_2_MS','MOS_3_4_MS','MOS_5_6_MS'])].copy()
    else:q=C.loc[C.group.eq(scope)].copy()
    q['eval_id']=q.group.astype(str)+'|'+q.unit_key.astype(str)
    for fold in (0,1,2):
        train=q.loc[q.fold.ne(fold)].copy()
        test=q.loc[q.fold.eq(fold)].copy()
        # NB02 predictions are present for held-out test samples ONLY.
        assert set(test.fold)=={fold} if len(test) else True
        held=units.loc[units.group.isin(q.group.unique())&units.fold.eq(fold)]
        other=units.loc[units.group.isin(q.group.unique())&units.fold.ne(fold)]
        if len(held) and len(other) and len(train):
            buffered=unary_union([z.buffer(GUARD_M) for z in held.full_geometry])
            ix=other.full_geometry.map(lambda g:g.intersects(buffered)).to_numpy(bool)
            bad=other.loc[ix]
            removed=set(bad.group.astype(str)+'|'+bad.unit_key.astype(str))
            train=train.loc[~train.eval_id.isin(removed)].copy()
        valid=(len(train)>=50 and len(test)>=30 and
               train.eval_id.nunique()>=5 and test.eval_id.nunique()>=2 and
               train.label.nunique()==2 and test.label.nunique()==2)
        if not valid:
            RESULTS.append(dict(scope=scope,fold=fold,status='insufficient independent units/classes',
                train_units=train.eval_id.nunique(),val_units=test.eval_id.nunique()))
            print('NB02 BRIDGE SKIP',scope,fold,flush=True)
            continue
        assert set(train.eval_id).isdisjoint(set(test.eval_id))
        # Strict: model trained from MS on other folds, with zero NB02
        # training predictions used. NB02 test checkpoint did not see test labels.
        ms_prob=train_ms_classifier(train,test)
        nb_prob=test.nb02_prob.to_numpy(float)
        fused=(1.-ALPHA_MS)*nb_prob+ALPHA_MS*ms_prob
        score={v:metrics(test,p) for v,p in [('NB02_RGB',nb_prob),
             ('MS_relative_only',ms_prob),('NB02_plus_MS_fixed25',fused)]}
        RESULTS.append(dict(scope=scope,fold=fold,status='evaluated',
             train_units=train.eval_id.nunique(),val_units=test.eval_id.nunique(),
             n_train=len(train),n_val=len(test),
             auc_nb02=score['NB02_RGB']['auc'],
             auc_ms_only=score['MS_relative_only']['auc'],
             auc_fusion=score['NB02_plus_MS_fixed25']['auc'],
             delta_auc=score['NB02_plus_MS_fixed25']['auc']-score['NB02_RGB']['auc'],
             brier_nb02=score['NB02_RGB']['brier'],
             brier_fusion=score['NB02_plus_MS_fixed25']['brier']))
        print('NB02 BRIDGE',scope,fold,
            'RGB AUC',round(score['NB02_RGB']['auc'],3),
            'MS AUC',round(score['MS_relative_only']['auc'],3),
            'Fixed25 AUC',round(score['NB02_plus_MS_fixed25']['auc'],3),flush=True)
RESULT=pd.DataFrame(RESULTS)
RESULT.to_csv(PROB_DIR/'08_v011_nb02_ms_paired_spatial_auc.csv',index=False)
display(RESULT.round(3))
evals=RESULT.loc[RESULT.status.eq('evaluated')]
if not evals.empty:
    display(evals.groupby('scope').agg(
         folds=('fold','size'),auc_nb02=('auc_nb02','mean'),
         auc_ms_only=('auc_ms_only','mean'),auc_fusion=('auc_fusion','mean'),
         delta_auc=('delta_auc','mean')).round(3))
print('This is unit-balanced point AUC/Brier, NOT NB02 full-mask F1/IoU or cover R².')


## 09 — Seven-channel RGB + MS U-Net

Train a genuine seven-channel U-Net at the **RGB reference pixel grid**. The four MS layers are red, green, red-edge and NIR, normalized as four relative proportions per pixel. We preserve all valid imagery and mark unavailable MS pixels as invalid training/evaluation support; absent MS pixels are **not negative examples**. This experiment reuses NB02 source-labelled RGB caches and NB03's frozen **inner spatial split** rather than creating random validation folds.

Training uses the original outer fold assignments and inner checkpoint selection. The frozen RGB model is an eligible selection fallback. **No full-cover superiority claim** will be made from the initial chip-level paired segmentation results.


### 09A — Strict paired RGB/MS chips


In [ ]:
# 09A — Full-resolution 7-channel chips, resumable by acquisition and fold
# Native NB02 RGB source-labelled cache + NB03 strict inner split, both read-only.
# Runs independently of 06B/07A/08A. Read only 256x256 RGB-aligned MS windows.
import sys, subprocess, importlib.util, gc, os, json, hashlib
from pathlib import Path
from collections import defaultdict
from contextlib import ExitStack
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.windows import Window
import torch
if importlib.util.find_spec('segmentation_models_pytorch') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','segmentation-models-pytorch'])
import segmentation_models_pytorch as smp
ROOT_U=ROOT
U_OUT=OUT/'09_unet_rgb_fourband'
U_OUT.mkdir(parents=True,exist_ok=True)
U_CACHE=ROOT_U/'Outputs/v4/04_ms_unet_fourband_chip_cache'
U_CACHE.mkdir(parents=True,exist_ok=True)
RGB_CACHE=ROOT_U/'Outputs/v4/02_rgb_unet_field78_v4_003_full/chip_cache'
NB03_SPLITS=ROOT_U/'Outputs/v4/03_rgb_dsm_multiscale_v4_007'
RGB_CKPT={i:ROOT_U/'Outputs/v4/02_rgb_unet_field78_v4_007_full'/f'NB02_V4_007_fold{i}_best.pt' for i in (0,1,2)}
assert all(p.is_file() for p in RGB_CKPT.values()),'NB02 frozen checkpoint missing'
MS_FOLDS=(0,1,2);CHIP_U=256
MS_GROUPS=['Baego_MS','MOS_1_2_MS','MOS_3_4_MS','MOS_5_6_MS',
           'Yangambi_YBJoseph_right_MS','Yangambi_YBForgeo_right_MS']
MS_BY_MOSAIC=defaultdict(list)
for group in MS_GROUPS:MS_BY_MOSAIC[MOSAIC_KEY[group]].append(group)
# Validate the complete source-labelled cache contract. Do not silently
# substitute old RGB chips with different label or fold semantics.
def get_rgb_cache(fold,role):
    paths=sorted(RGB_CACHE.glob(f'field78_{role}_fold{fold}_*.npz'))
    if not paths:
        # NB03 writes compatible source-labelled copies if source is missing.
        alt=ROOT_U/'Outputs/v4/03_rgb_dsm_multiscale_v4_007/chip_cache'
        paths=sorted(alt.glob(f'field78_{role}_fold{fold}_*.npz'))
    assert len(paths)==1,(fold,role,'Expected one fingerprinted source-labelled RGB cache',list(map(str,paths)))
    with np.load(paths[0],allow_pickle=False) as z:
        needed={'rgb','y','w','source','site','mosaic','row','col','key'}
        assert needed.issubset(z.files),(paths[0],z.files)
        fields={k:z[k] for k in needed}
    n=len(fields['rgb'])
    assert fields['rgb'].shape==(n,CHIP_U,CHIP_U,3)
    assert all(fields[k].shape[0]==n for k in ('y','w','source','mosaic','row','col'))
    items=[]
    for i in range(n):
        src=fields['source'][i]
        assert np.array_equal(src>0,fields['w'][i]>0),(fold,role,i,'source-mask mismatch')
        items.append(dict(rgb=fields['rgb'][i],y=fields['y'][i],w=fields['w'][i],
            source=src,site=str(fields['site'][i]),mosaic=str(fields['mosaic'][i]),
            row=int(fields['row'][i]),col=int(fields['col'][i])))
    print('READ-ONLY NB02 RGB',role,'fold',fold,len(items),'chips',flush=True)
    return items,paths[0]
def selected_inner_roles(fold):
    p=NB03_SPLITS/f'03_v4_002_inner_spatial_split_fold{fold}.csv'
    assert p.is_file(),('Strict NB03 inner split audit required; do not substitute random validation',p)
    x=pd.read_csv(p)
    assert {'mosaic','row','col','inner_role'}.issubset(x.columns)
    assert not x.duplicated(['mosaic','row','col']).any()
    return {(str(r.mosaic),int(r.row),int(r.col)):str(r.inner_role) for r in x.itertuples(index=False)}
def source_stamp(paths):
    vals=[]
    for p in sorted(set(map(Path,paths)),key=str):
        stat=p.stat();vals.append([str(p),int(stat.st_size),int(stat.st_mtime_ns)])
    return vals
# A single VRT is created per MS band/acquisition, reused for all chips.
def open_vrts(rgbds,group,stack):
    out={}
    for band in ('red','green','red_edge','nir'):
        src=stack.enter_context(rasterio.open(BANDS[group][band]))
        ix={'red':1,'green':2,'red_edge':3,'nir':4}[band] if group=='Baego_MS' else 1
        # Explicit destination matches NB02 RGB grid. Float32 VRT avoids
        # interpolation rounding in band digital-number signals.
        vrt=stack.enter_context(WarpedVRT(src,crs=rgbds.crs,
             transform=rgbds.transform,width=rgbds.width,height=rgbds.height,
             resampling=Resampling.bilinear,dtype='float32'))
        out[band]=(vrt,ix)
    return out
def make_spectral_chip(vrts,win):
    v=[];m=[]
    for band in ('red','green','red_edge','nir'):
        ds,idx=vrts[band]
        a=ds.read(idx,window=win,masked=True,out_dtype='float32')
        z=np.asarray(a.filled(0),np.float32)
        ok=(~np.ma.getmaskarray(a))&np.isfinite(z)&(z>0)
        v.append(np.where(ok,z,0));m.append(ok)
    signal=np.stack(v)
    good=np.logical_and.reduce(m)
    total=signal.sum(axis=0)
    good &= total>1e-6
    # Four relative spectral channels, not calibrated reflectance indices.
    rel=np.where(good[None,:,:],signal/np.maximum(total[None,:,:],1e-6),0)
    assert rel.shape==(4,CHIP_U,CHIP_U)
    return rel.astype(np.float16),good.astype(np.uint8)
def create_matched(fold,role):
    allitems,rgbsrc=get_rgb_cache(fold,role)
    inner=selected_inner_roles(fold) if role=='train' else None
    if inner is not None:
        accepted={'inner_train','inner_val_candidate'}
        allitems=[q for q in allitems if inner.get((q['mosaic'],q['row'],q['col'])) in accepted]
        assert len(allitems)>=100,('Too few NB03-matched inner chips',fold,len(allitems))
    bymosaic=defaultdict(list)
    for q in allitems:
        if q['mosaic'] in MS_BY_MOSAIC:bymosaic[q['mosaic']].append(q)
    manifest=[]
    assert bymosaic,('No RGB chips intersect MS mapped mosaics',fold,role)
    for mosaic,items in sorted(bymosaic.items()):
        groups=MS_BY_MOSAIC[mosaic]
        paths=[rgbsrc,DATA/'Ortho_RGB'/mosaic]
        for group in groups:paths.extend(BANDS[group].values())
        fingerprint=dict(version='NB04_V4_013_rel4_nativeRGBgrid_v1',
              fold=fold,role=role,mosaic=mosaic,
              sources=source_stamp(paths),
              coords=[(q['row'],q['col']) for q in items],
              split_audit=source_stamp([NB03_SPLITS/f'03_v4_002_inner_spatial_split_fold{fold}.csv']) if inner else [])
        sig=hashlib.sha256(json.dumps(fingerprint,sort_keys=True).encode()).hexdigest()
        destination=U_CACHE/f'fold{fold}_{role}_{sig[:24]}.npz'
        valid_file=False
        if destination.is_file():
            try:
                with np.load(destination,allow_pickle=False) as z:
                    valid_file=(str(z['signature'].item())==sig and
                       z['ms'].shape[1:]==(4,CHIP_U,CHIP_U) and
                       len(z['ms'])==len(z['rgb'])==len(z['y']))
            except Exception:valid_file=False
        if valid_file:
            print('MS CHIP CACHE HIT',fold,role,mosaic,flush=True)
            manifest.append(dict(fold=fold,role=role,mosaic=mosaic,path=str(destination),signature=sig));continue
        print('EXTRACT MS CHIPS',fold,role,mosaic,len(items),flush=True)
        matched=[]
        with rasterio.Env(GDAL_CACHEMAX=256):
            with ExitStack() as stack:
                rgb=stack.enter_context(rasterio.open(DATA/'Ortho_RGB'/mosaic))
                group_vrts={group:open_vrts(rgb,group,stack) for group in groups}
                for j,q in enumerate(sorted(items,key=lambda a:(a['row'],a['col']))):
                    window=Window(q['col'],q['row'],CHIP_U,CHIP_U)
                    candidates=[(group,*make_spectral_chip(group_vrts[group],window)) for group in groups]
                    # For Yangambi two acquisitions can be present on the same
                    # RGB mosaic. Pick the one with greatest four-band coverage.
                    group,ms4,valid=max(candidates,key=lambda z:int(z[2].sum()))
                    source=q['source'].copy()
                    w=(q['w'].astype(bool)&valid.astype(bool)).astype(np.uint8)
                    source[~w.astype(bool)]=0
                    if int(w.sum())<50:continue
                    matched.append(dict(rgb=q['rgb'],y=q['y'],w=w,source=source,
                        ms=ms4,mosaic=mosaic,site=q['site'],row=q['row'],col=q['col'],
                        group=group,inner_role=inner[(mosaic,q['row'],q['col'])] if inner else 'outer_val'))
                    if (j+1)%100==0:print('  ',mosaic,j+1,'/',len(items),flush=True)
        if not matched:
            print('NO ELIGIBLE FOUR-BAND CHIPS',fold,role,mosaic,flush=True);continue
        temp=destination.with_suffix('.partial.npz')
        np.savez_compressed(temp,signature=np.array(sig),
          rgb=np.stack([q['rgb'] for q in matched]).astype('uint8'),
          ms=np.stack([q['ms'] for q in matched]).astype('float16'),
          y=np.stack([q['y'] for q in matched]).astype('uint8'),
          w=np.stack([q['w'] for q in matched]).astype('uint8'),
          source=np.stack([q['source'] for q in matched]).astype('uint8'),
          site=np.array([q['site'] for q in matched],dtype='U32'),
          group=np.array([q['group'] for q in matched],dtype='U48'),
          mosaic=np.array([q['mosaic'] for q in matched],dtype='U128'),
          row=np.array([q['row'] for q in matched],dtype='int32'),
          col=np.array([q['col'] for q in matched],dtype='int32'),
          inner_role=np.array([q['inner_role'] for q in matched],dtype='U48'))
        os.replace(temp,destination)
        print('MS CHIP SAVED',fold,role,mosaic,len(matched),flush=True)
        manifest.append(dict(fold=fold,role=role,mosaic=mosaic,
                        path=str(destination),signature=sig))
    assert manifest,('No valid four-band matched chip caches',fold,role)
    return manifest
MS_MANIFEST=[]
for fold in MS_FOLDS:
    for role in ('train','val'):
        MS_MANIFEST.extend(create_matched(fold,role))
MS_MANIFEST=pd.DataFrame(MS_MANIFEST)
MS_MANIFEST.to_csv(U_OUT/'09_matched_chip_manifest.csv',index=False)
display(MS_MANIFEST[['fold','role','mosaic','path']])
print('09A DONE. Read-only original RGB caches and original rasters unchanged.')


### 09B — NB02-initialized seven-channel training


In [ ]:
# 09B — Train genuine seven-input-channel U-Net for each frozen outer fold.
# First 3 channels reuse pretrained NB02 RGB encoder weights; last 4 start
# EXACTLY zero, so zero epochs reproduces original NB02 RGB logits.
# Train ONLY source-labelled NB03 inner-training chips, select on exhaustive
# original NB03 inner-validation chips, never on outer held-out samples.
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
import torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, torch, copy, gc, os, json
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type=='cuda','Use Colab GPU for 09B'
torch.manual_seed(20261117)
torch.backends.cudnn.benchmark=True
BATCH=6;EPOCHS=14;TRAIN_STEPS=90;PATIENCE=4
LR=8e-5;SEED=20261117
CKPT_U={fold:U_OUT/f'NB04_V4_013_RGB_MS_UNet_fold{fold}_best.pt' for fold in MS_FOLDS}
def read_paired_items(fold,role):
    rows=MS_MANIFEST.loc[MS_MANIFEST.fold.eq(fold)&MS_MANIFEST.role.eq(role)]
    result=[]
    assert not rows.empty,(fold,role)
    for row in rows.itertuples(index=False):
        with np.load(row.path,allow_pickle=False) as z:
            assert str(z['signature'].item())==row.signature
            arrays={k:z[k] for k in ['rgb','ms','y','w','source','site',
                                    'group','mosaic','row','col','inner_role']}
        for i in range(len(arrays['y'])):
            result.append({k:(str(v[i]) if k in ('site','group','mosaic','inner_role')
                     else v[i]) for k,v in arrays.items()})
    assert result
    return result
class PairedDataset(Dataset):
    def __init__(self,items,augment=False):
        self.items=items;self.augment=augment
    def __len__(self):return len(self.items)
    def __getitem__(self,i):
        q=self.items[i]
        rgb=q['rgb'].astype(np.float32).transpose(2,0,1)/255.
        ms=q['ms'].astype(np.float32)
        x=np.concatenate([rgb,ms],axis=0)
        y=q['y'][None].astype('float32')
        w=q['w'][None].astype('float32')
        if self.augment:
            k=int(torch.randint(0,4,(1,)).item())
            x=np.rot90(x,k,axes=(1,2));y=np.rot90(y,k,axes=(1,2));w=np.rot90(w,k,axes=(1,2))
            if torch.rand(1).item()<.5:
                x=x[:,:,::-1];y=y[:,:,::-1];w=w[:,:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)),
                torch.from_numpy(np.ascontiguousarray(w)),q['site'])
def make_unet(fold):
    state=torch.load(RGB_CKPT[fold],map_location='cpu',weights_only=False)
    assert state['version']=='V4.007' and int(state['fold'])==fold
    original=smp.Unet(encoder_name='resnet18',encoder_weights=None,
                      in_channels=3,classes=1)
    original.load_state_dict({k[4:]:v for k,v in state['model'].items()},strict=True)
    fused=smp.Unet(encoder_name='resnet18',encoder_weights=None,
                   in_channels=7,classes=1)
    assert original.encoder.conv1.weight.shape[1]==3
    assert fused.encoder.conv1.weight.shape[1]==7
    weights=original.state_dict()
    new=copy.deepcopy(weights)
    kernel=weights['encoder.conv1.weight']
    extra=torch.zeros((kernel.shape[0],4,*kernel.shape[2:]),dtype=kernel.dtype)
    new['encoder.conv1.weight']=torch.cat([kernel,extra],dim=1)
    fused.load_state_dict(new,strict=True)
    assert torch.count_nonzero(fused.encoder.conv1.weight[:,3:])==0
    return original.to(DEVICE).eval().requires_grad_(False),fused.to(DEVICE)
def epoch(model,loader,optimizer=None,baseline=False):
    training=optimizer is not None
    model.train(training)
    total={};site_totals={}
    scaler=None
    for i,(x,y,w,sites) in enumerate(loader):
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE);w=w.to(DEVICE)
        if baseline:x=x[:,:3]
        if w.sum()==0:continue
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                z=model(x)
            pixel=F.binary_cross_entropy_with_logits(z.float(),y,reduction='none')
            objective=(pixel*w).sum()/w.sum().clamp_min(1.)
            if training:
                optimizer.zero_grad(set_to_none=True)
                objective.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),2.)
                optimizer.step()
        for j,site in enumerate(sites):
            ww=w[j:j+1];vv=float(ww.sum().item())
            if vv<=0:continue
            loss=float((pixel[j:j+1]*ww).sum().item()/vv)
            site_totals.setdefault(str(site),[]).append(loss)
    return float(np.mean([np.mean(v) for v in site_totals.values()])) if site_totals else np.inf
def train_select(fold):
    allitems=read_paired_items(fold,'train')
    train=[q for q in allitems if q['inner_role']=='inner_train']
    val=[dict(q,w=(q['w'].astype(bool)&(q['source']>=2)).astype('uint8'))
         for q in allitems if q['inner_role']=='inner_val_candidate']
    val=[q for q in val if np.any(q['w'])]
    assert len(train)>=80 and len(val)>=8,('Insufficient matched inner chips',fold,len(train),len(val))
    assert all(not np.any((q['source']==1)&(q['w']>0)) for q in val)
    print('09B INNER',fold,'train',len(train),'selection',len(val),flush=True)
    counts=pd.Series([q['site'] for q in train]).value_counts().to_dict()
    weights=np.array([((2.5 if np.any(q['y']&q['w']) else 1.)/counts[q['site']])
                       for q in train],np.float64)
    sampler=WeightedRandomSampler(weights,num_samples=TRAIN_STEPS*BATCH,replacement=True)
    trainloader=DataLoader(PairedDataset(train,True),sampler=sampler,
        batch_size=BATCH,num_workers=0,pin_memory=True)
    valloader=DataLoader(PairedDataset(val),shuffle=False,
        batch_size=BATCH,num_workers=0,pin_memory=True)
    rgb,model=make_unet(fold)
    # Early fusion MUST exactly reproduce pretrained RGB before adaptation.
    # Both nets in eval mode: BN running statistics must match exactly.
    model.eval()
    with torch.inference_mode():
        x=torch.from_numpy(np.random.default_rng(3).random((1,7,64,64)).astype('float32')).to(DEVICE)
        a=rgb(x[:,:3]);b=model(x)
        delta=float((a-b).abs().max())
        assert delta<1e-5,('Zero MS init does not reproduce frozen RGB',delta)
    baseline=epoch(rgb,valloader,baseline=True)
    initial=epoch(model,valloader)
    assert abs(initial-baseline)<1e-5,('Initial paired BCE mismatch',baseline,initial)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    best=float(initial);best_state=copy.deepcopy({k:v.detach().cpu().clone()
                   for k,v in model.state_dict().items()})
    wait=0;best_ep=0;history=[]
    for ep in range(1,EPOCHS+1):
        train_bce=epoch(model,trainloader,optimizer)
        val_bce=epoch(model,valloader)
        history.append(dict(fold=fold,epoch=ep,train_macro_BCE=train_bce,
                inner_macro_BCE=val_bce,baseline_inner_BCE=baseline))
        print('09B FOLD',fold,'epoch',ep,'train',round(train_bce,4),
              'inner',round(val_bce,4),'RGB',round(baseline,4),flush=True)
        if np.isfinite(val_bce) and val_bce<best-1e-4:
            best=val_bce;best_ep=ep;wait=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            wait+=1
            if wait>=PATIENCE:break
    selected=(best_ep>0 and best<baseline-1e-4)
    checkpoint=dict(version='V4.013',fold=fold,architecture='smp.Unet_resnet18_in7_rel4',
           base_checkpoint=str(RGB_CKPT[fold]),state=best_state,
           baseline_inner_BCE=baseline,selected_inner_BCE=best,
           selected_epoch=best_ep,use_fusion=selected,
           train_chips=len(train),selection_chips=len(val),
           protocol='frozen_NB03_source_exhaustive_inner_guard_v4_007',
           bands=['RGB_R','RGB_G','RGB_B','relative_MS_red','relative_MS_green',
                  'relative_MS_red_edge','relative_MS_nir'])
    target=CKPT_U[fold];tmp=target.with_suffix('.tmp.pt')
    torch.save(checkpoint,tmp);os.replace(tmp,target)
    pd.DataFrame(history).to_csv(U_OUT/f'09_unet_inner_history_fold{fold}.csv',index=False)
    print('09B COMPLETED',fold,'SELECT FUSION',selected,'best epoch',best_ep,flush=True)
    del trainloader,valloader,allitems,train,val,rgb,model,optimizer
    gc.collect();torch.cuda.empty_cache()
for fold in MS_FOLDS:
    if CKPT_U[fold].is_file():
        ck=torch.load(CKPT_U[fold],map_location='cpu',weights_only=False)
        assert ck['version']=='V4.013' and ck['fold']==fold and ck['base_checkpoint']==str(RGB_CKPT[fold])
        print('09B COMPLETED CHECKPOINT HIT',fold,'selected',ck['use_fusion'],flush=True)
    else:train_select(fold)
print('09B complete: three frozen-outer-fold 7-channel U-Net candidates available.')


### 09C — Paired held-out segmentation check


In [ ]:
# 09C — Paired held-out *CHIP* pixel F1/IoU and BCE, identical valid-MS mask
# Outer-held-out labelled RGB chips only; no fold-specific threshold tuning.
# These are NOT full field plot/circle cover R², and overlapping chips may
# duplicate pixels; therefore diagnostics only, not final paper estimates.
from collections import defaultdict
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
F1_ROWS=[]
EVAL_BATCH=6
for fold in MS_FOLDS:
    rgb,model=make_unet(fold)
    ck=torch.load(CKPT_U[fold],map_location='cpu',weights_only=False)
    assert ck['version']=='V4.013' and int(ck['fold'])==fold
    assert ck['base_checkpoint']==str(RGB_CKPT[fold])
    model.load_state_dict(ck['state'],strict=True)
    rgb.eval().requires_grad_(False)
    model.eval().requires_grad_(False)
    use_fusion=bool(ck['use_fusion'])
    items=read_paired_items(fold,'val')
    print('09C FOLD',fold,'validation chips',len(items),'selected MS',use_fusion,flush=True)
    loader=DataLoader(PairedDataset(items),batch_size=EVAL_BATCH,
                       shuffle=False,num_workers=0,pin_memory=True)
    sums=defaultdict(lambda:defaultdict(lambda:dict(tp=0,fp=0,fn=0,tn=0,
                                                     count=0,bce_sum=0.)))
    with torch.inference_mode():
        at=0
        for x,y,w,sites in loader:
            n=x.shape[0];x=x.to(DEVICE);y=y.to(DEVICE);w=w.to(DEVICE)
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                z0=rgb(x[:,:3]);z1=model(x) if use_fusion else z0
            for j in range(n):
                for variant,logit in [('RGB',z0[j]),('RGB_MS_Selected',z1[j])]:
                    mask=w[j].bool();target=y[j].bool()
                    if not mask.any():continue
                    p=torch.sigmoid(logit)>0.5
                    stats=sums[str(sites[j])][variant]
                    stats['tp']+=int((p&target&mask).sum())
                    stats['fp']+=int((p&~target&mask).sum())
                    stats['fn']+=int((~p&target&mask).sum())
                    stats['tn']+=int((~p&~target&mask).sum())
                    stats['count']+=int(mask.sum())
                    stats['bce_sum']+=float((F.binary_cross_entropy_with_logits(
                        logit.float(),y[j],reduction='none')*w[j]).sum())
            at+=n
            if at%120<EVAL_BATCH:print('09C',fold,at,'/',len(items),flush=True)
    for site,v in sums.items():
        for variant,z in v.items():
            tp,fp,fn,tn=(z[k] for k in ('tp','fp','fn','tn'))
            F1_ROWS.append(dict(fold=fold,site=site,variant=variant,
                selected_fusion=use_fusion,n_valid_pixels=z['count'],
                TP=tp,FP=fp,FN=fn,TN=tn,
                F1=2*tp/max(2*tp+fp+fn,1),
                IoU=tp/max(tp+fp+fn,1),
                BCE=z['bce_sum']/max(z['count'],1)))
    del rgb,model,loader,items
    gc.collect();torch.cuda.empty_cache()
SEGMENT_QA=pd.DataFrame(F1_ROWS)
SEGMENT_QA.to_csv(U_OUT/'09_v013_matched_outer_chip_segmentation_diagnostics.csv',index=False)
display(SEGMENT_QA.round(3))
paired=SEGMENT_QA.pivot_table(index=['fold','site'],columns='variant',values=['F1','IoU','BCE'])
if len(paired):
    display(paired.round(3))
print('09C is a paired held-out CHIP check only. Original plot/circle full-cover R², MAE and >40% MOS bias remain next-stage metrics.')


## 10 — Interpretation and next-stage fractional cover

**What 09A–09C establish:** Can a genuine **RGB + four-band MS U-Net**, initialized from the frozen NB02 RGB U-Net, be optimized on the same source-labelled training chips and improve *held-out MS-valid* segmentation metrics? The original pretrained RGB network remains an eligible fallback, with selection based solely on disjoint inner-validation regions.

**What 09A–09C do not establish:** These initial val-chip metrics do not measure each original plot/circle exactly once; windows overlap, support can be incomplete, and units may be sampled unequally. **The next NB04 version** should perform dense, Hann-blended paired inference on original exhaustive field plots and circles over exactly identical RGB+MS support, then compare NB02 vs seven-channel U-Net against the previously reported NB02 and NB03 fractions and error patterns, focusing on MOS observed cover >40%.

**QA priority:** Check RGB and MS crown-scale coregistration using visibly corresponding features before accepting seven-channel segmentation as scientifically sound. Bilinear VRT resampling does not improve native MS spatial resolution; uncalibrated MS band ratios reflect proportional digital numbers, not physically calibrated reflectances.
